In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from src.config import CACHE

features = pd.read_pickle(CACHE / "features_deriv_4s.pkl")

# keep only AD and CN subjects
data = features[features.label.isin(["AD", "CN"])].reset_index(drop=True)
X = data.drop(columns=["subject", "label"]).values      # the 114 features
y = (data.label == "AD").astype(int).values             # AD = 1, CN = 0
groups = data.subject.values                            # who each epoch belongs to

# 5 folds, and all epochs of one subject always stay together
folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))

fold_scores = []
for train, test in folds.split(X, y, groups):
    model.fit(X[train], y[train])                                   # learn from training epochs only

    epoch_prob = model.predict_proba(X[test])[:, 1]                 # chance of AD for each test epoch
    table = pd.DataFrame({"subject": groups[test], "prob": epoch_prob, "true": y[test]})
    per_subject = table.groupby("subject").mean()                   # average over each subject's epochs

    guess = (per_subject.prob > 0.5).astype(int)                    # AD if average chance is over 50%
    fold_scores.append((guess == per_subject.true.round()).mean())  # fraction of subjects correct

print("accuracy per fold:", np.round(fold_scores, 2))
print("mean:", round(np.mean(fold_scores), 2), "+/-", round(np.std(fold_scores), 2))

accuracy per fold: [0.85 0.69 0.79 0.92 0.77]
mean: 0.8 +/- 0.08


In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from src.evaluate import subject_cv

def make_logreg():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))

accuracies = []
for seed in range(10):                                   # 10 different shuffles
    res = subject_cv(features, make_logreg, ["AD", "CN"], seed)
    accuracies.append((res.true == res.guess).mean())    # fraction of all 65 subjects correct

print(np.round(accuracies, 2))
print("mean:", round(np.mean(accuracies), 3), "+/-", round(np.std(accuracies), 3))

[0.77 0.77 0.83 0.78 0.78 0.82 0.78 0.8  0.8  0.8 ]
mean: 0.794 +/- 0.018


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

def make_rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)

models = {"LogReg": make_logreg, "RandomForest": make_rf}
tasks = [["AD", "CN"], ["AD", "FTD", "CN"]]

for classes in tasks:
    for name, make_model in models.items():
        accuracies = []
        for seed in range(5):                                    # 5 shuffles, to keep it quick
            res = subject_cv(features, make_model, classes, seed)
            accuracies.append((res.true == res.guess).mean())
        print("-".join(classes), "|", name, "| mean accuracy:", round(np.mean(accuracies), 3))
        print(confusion_matrix(res.true, res.guess))             # last shuffle: rows = true, columns = guessed
        print()

AD-CN | LogReg | mean accuracy: 0.788
[[28  8]
 [ 6 23]]

AD-CN | RandomForest | mean accuracy: 0.831
[[30  6]
 [ 4 25]]

AD-FTD-CN | LogReg | mean accuracy: 0.564
[[20 12  4]
 [10  6  7]
 [ 2  5 22]]

AD-FTD-CN | RandomForest | mean accuracy: 0.645
[[28  1  7]
 [16  5  2]
 [ 4  2 23]]



In [4]:
from src.evaluate import repeat_cv
from src.config import RESULTS

rows = []
for classes in tasks:
    for name, make_model in models.items():
        row = repeat_cv(features, make_model, classes)
        row.update({"task": "-".join(classes), "model": name})
        rows.append(row)

table = pd.DataFrame(rows).round(3)
print(table)
table.to_csv(RESULTS / "baseline_results.csv", index=False)

   accuracy  macro_f1  recall_AD  recall_CN       task         model  \
0     0.794     0.792      0.794      0.793      AD-CN        LogReg   
1     0.822     0.820      0.808      0.838      AD-CN  RandomForest   
2     0.565     0.538      0.581      0.766  AD-FTD-CN        LogReg   
3     0.644     0.588      0.775      0.821  AD-FTD-CN  RandomForest   

   recall_FTD  
0         NaN  
1         NaN  
2       0.287  
3       0.217  


In [5]:
pairs = [["AD", "FTD"], ["FTD", "CN"]]

rows = []
for classes in pairs:
    for name, make_model in models.items():
        row = repeat_cv(features, make_model, classes)
        row.update({"task": "-".join(classes), "model": name})
        rows.append(row)

print(pd.DataFrame(rows).round(3))

   accuracy  macro_f1  recall_AD  recall_FTD    task         model  recall_CN
0     0.524     0.493      0.631       0.357  AD-FTD        LogReg        NaN
1     0.634     0.519      0.919       0.187  AD-FTD  RandomForest        NaN
2     0.727     0.721        NaN       0.652  FTD-CN        LogReg      0.786
3     0.773     0.764        NaN       0.652  FTD-CN  RandomForest      0.869


In [6]:
from src.features import regional_table

regional = regional_table(features)
print(regional.shape)                        # expect (17419, 32): 30 features + subject + label

rows = []
for classes in [["AD", "FTD"], ["AD", "FTD", "CN"]]:
    for name, make_model in models.items():
        row = repeat_cv(regional, make_model, classes)
        row.update({"task": "-".join(classes), "model": name})
        rows.append(row)

print(pd.DataFrame(rows).round(3))

(17419, 32)
   accuracy  macro_f1  recall_AD  recall_FTD       task         model  \
0     0.554     0.512      0.694       0.335     AD-FTD        LogReg   
1     0.617     0.521      0.872       0.217     AD-FTD  RandomForest   
2     0.561     0.530      0.622       0.257  AD-FTD-CN        LogReg   
3     0.619     0.574      0.733       0.243  AD-FTD-CN  RandomForest   

   recall_CN  
0        NaN  
1        NaN  
2      0.728  
3      0.776  


In [7]:
# one row per person: average each feature over that person's epochs
subject_means = regional.groupby(["subject", "label"]).mean().reset_index()

ad = subject_means[subject_means.label == "AD"].drop(columns=["subject", "label"])
ftd = subject_means[subject_means.label == "FTD"].drop(columns=["subject", "label"])

# effect size: difference between group means, in units of the spread
spread = np.sqrt((ad.var() + ftd.var()) / 2)
effect = ((ad.mean() - ftd.mean()) / spread).round(2)

# the 10 features with the biggest difference (positive = higher in AD)
print(effect.reindex(effect.abs().sort_values(ascending=False).index).head(10))

frontal_theta            0.56
frontal_beta            -0.45
frontal_gamma           -0.45
temporal_beta           -0.45
temporal_theta/alpha     0.43
temporal_gamma          -0.41
occipital_theta/alpha    0.39
frontal_theta/alpha      0.35
temporal_delta           0.32
central_gamma           -0.28
dtype: float64
